# 페르소나 가상 투표 실험 — 1인 파일럿 (LangChain + OpenAI)

`nvidia/Nemotron-Personas-Korea`에서 1명을 랜덤 샘플링해, 2026-04 한국 정치 상황 컨텍스트를 주고 OpenAI 모델에게 투표 의향과 이유를 받는다.

**저장 컬럼**: `persona_uuid, vote, reason, model, elapsed_sec`

## 0. 사전 준비

```bash
pip install langchain langchain-openai langchain-ollama python-dotenv pyarrow pandas
```

프로젝트 루트 또는 `backend/`에 `.env.local` 생성:
```
OPENAI_API_KEY=sk-...
```

OpenAI 모델은 [Cell 1]의 `MODEL` 변수에서, 로컬(Ollama) 모델은 [Cell 10~12]의 `model_name` 인자에서 직접 지정.

In [ ]:
# [Cell 1] imports & env
import os, glob, json, time, random
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 노트북이 있는 폴더 (= backend/) 기준 경로
BASE = Path.cwd() if Path.cwd().name == "backend" else Path.cwd() / "backend"
DATA_DIR = BASE / "nvidia-personas" / "data"
CONTEXT_PATH = BASE / "context" / "voter_context.md"
RESULTS_PATH = BASE / "vote_results_all.csv"

# OpenAI 기본 모델 — Cell 9에서 사용
MODEL = "gpt-5.4-mini"

# .env.local 로드 — backend/.env.local 우선, 없으면 프로젝트 루트
for env_path in [BASE / ".env.local", BASE.parent / ".env.local"]:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"loaded: {env_path}")
        break

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 환경변수 필요 — .env.local 확인"
print(f"openai model: {MODEL}")
print(f"BASE: {BASE}")

In [ ]:
# [Cell 2] load voter context
POLITICAL_CONTEXT = CONTEXT_PATH.read_text(encoding="utf-8")
print(POLITICAL_CONTEXT[:500], "...")

In [ ]:
# [Cell 3] sample 1 persona
# 9개 샤드 중 1개 랜덤 선택 → 그 안에서 1행 랜덤 샘플
shards = sorted(glob.glob(str(DATA_DIR / "train-*.parquet")))
shard = random.choice(shards)
df = pq.read_table(shard).to_pandas()
row = df.sample(n=1).iloc[0]

persona_uuid = row["uuid"]
print(f"shard:    {Path(shard).name}")
print(f"uuid:     {persona_uuid}")
print(f"성/연령:  {row['sex']} / {row['age']}세")
print(f"지역:     {row['province']} {row['district']}")
print(f"직업:     {row['occupation']}")
print(f"학력:     {row['education_level']}")
print(f"\n[persona]\n{row['persona']}")

In [ ]:
# [Cell 4] persona card builder
def persona_card(r) -> str:
    fields = [
        ("성별", r["sex"]), ("연령", f"{r['age']}세"),
        ("혼인상태", r["marital_status"]),
        ("가구형태", r["family_type"]), ("주거", r["housing_type"]),
        ("학력", r["education_level"]), ("전공", r["bachelors_field"]),
        ("직업", r["occupation"]),
        ("지역", f"{r['province']} {r['district']}"),
    ]
    demo = "\n".join(f"- {k}: {v}" for k, v in fields if v)
    narr = (
        f"[요약]\n{r['persona']}\n\n"
        f"[직업적 면모]\n{r['professional_persona']}\n\n"
        f"[가족 면모]\n{r['family_persona']}\n\n"
        f"[문화적 배경]\n{r['cultural_background']}\n\n"
        f"[관심사]\n{r['hobbies_and_interests']}\n\n"
        f"[목표]\n{r['career_goals_and_ambitions']}"
    )
    return f"## 인구통계\n{demo}\n\n## 인물 서사\n{narr}"

card = persona_card(row)
print(card[:800], "...")

In [ ]:
# [Cell 5] prompt template & chain builder (provider toggle)
SYSTEM_TEMPLATE = """당신은 사회조사 시뮬레이션을 위한 가상 응답자다. 주어진 페르소나의 인물 서사·인구통계·가치관·관심사를 바탕으로, 그 인물이 실제로 할 법한 정치 선택을 1인칭 시점에서 추론한다.

다음 규칙을 따른다:
1. 아래 [정치상황 컨텍스트]에 명시된 정당과 쟁점만 사용한다. 외부 정보·최신 추정 금지.
2. 페르소나의 연령·지역·직업·생애 맥락·가치관과 정합적인 선택을 한다. 단순히 인구통계 평균에 의존하지 말고 인물 서사를 우선한다.
3. 무당층·기권도 정당한 선택지다. 페르소나가 정치에 무관심하거나 기존 정당 모두에 거리감이 있으면 그렇게 답한다.
4. 출력은 반드시 다음 JSON 형식만 (코드블록·설명·머리말 일절 금지):
{{{{"vote": "<정당명 또는 '무당층/기권'>", "reason": "<해당 인물의 1인칭 시점으로 200~400자 이내 한국어 설명>"}}}}

## [정치상황 컨텍스트]
{political_context}
"""

USER_TEMPLATE = """다음 페르소나가 2026년 6월 3일 지방선거(또는 가까운 미래의 총선·대선)에서 어느 정당을 지지·투표할지 추론하시오.

{persona_card}

JSON만 출력."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("user", USER_TEMPLATE),
])

# provider별 chain 빌더 — Cell 8의 run_one(provider, model_name) 가 사용
def build_chain(provider: str, model_name: str):
    if provider == "openai":
        llm = ChatOpenAI(
            model=model_name,
            temperature=0.7,
            model_kwargs={"response_format": {"type": "json_object"}},
        )
    elif provider == "ollama":
        from langchain_ollama import ChatOllama
        llm = ChatOllama(
            model=model_name,
            temperature=0.7,
            format="json",  # ollama JSON 강제
        )
    else:
        raise ValueError(f"unknown provider: {provider}")
    return prompt | llm | JsonOutputParser()

# 같은 모델 chain 캐싱 (반복 호출 시 재생성 방지)
_chain_cache = {}
def get_chain(provider: str, model_name: str):
    key = (provider, model_name)
    if key not in _chain_cache:
        _chain_cache[key] = build_chain(provider, model_name)
    return _chain_cache[key]

# 기본 chain (Cell 6 디버깅 셀이 사용)
chain = get_chain("openai", MODEL)
print(f"default chain: openai / {MODEL}")

In [ ]:
# [Cell 6] invoke LLM
t0 = time.perf_counter()
parsed = chain.invoke({
    "political_context": POLITICAL_CONTEXT,
    "persona_card": card,
})
elapsed = time.perf_counter() - t0

print(f"[elapsed] {elapsed:.2f}s\n")
print("투표:", parsed["vote"])
print("이유:", parsed["reason"])

In [ ]:
# [Cell 7] save result
record = {
    "persona_uuid": persona_uuid,
    "sex": row["sex"],
    "age": int(row["age"]),
    "marital_status": row["marital_status"],
    "family_type": row["family_type"],
    "housing_type": row["housing_type"],
    "education_level": row["education_level"],
    "bachelors_field": row["bachelors_field"],
    "occupation": row["occupation"],
    "province": row["province"],
    "district": row["district"],
    "persona_summary": row["persona"],
    "vote": parsed["vote"],
    "reason": parsed["reason"],
    "model": MODEL,
    "elapsed_sec": round(elapsed, 3),
    "response_file": "",  # 디버깅용 셀이라 raw 파일 저장은 안 함
    "professional_persona": row["professional_persona"],
    "sports_persona": row["sports_persona"],
    "arts_persona": row["arts_persona"],
    "travel_persona": row["travel_persona"],
    "culinary_persona": row["culinary_persona"],
    "family_persona": row["family_persona"],
    "cultural_background": row["cultural_background"],
    "skills_and_expertise": row["skills_and_expertise"],
    "hobbies_and_interests": row["hobbies_and_interests"],
    "career_goals_and_ambitions": row["career_goals_and_ambitions"],
}

row_df = pd.DataFrame([record])
if RESULTS_PATH.exists():
    row_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")
else:
    row_df.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")

print(f"저장됨 → {RESULTS_PATH}\n")
pd.read_csv(RESULTS_PATH)

In [ ]:
# [Cell 8] one-shot helper — provider/model 인자 받음
def run_one(provider: str = "openai", model_name: str | None = None, verbose: bool = True):
    """1명 샘플 → 투표 추론 → CSV 누적 → (verbose=True면) 항목별 한 줄 출력"""
    if model_name is None:
        model_name = MODEL  # OpenAI 기본
    ch = get_chain(provider, model_name)

    # 1) sample
    shard = random.choice(shards)
    df_local = pq.read_table(shard).to_pandas()
    r = df_local.sample(n=1).iloc[0]

    # 2) invoke
    t0 = time.perf_counter()
    p = ch.invoke({
        "political_context": POLITICAL_CONTEXT,
        "persona_card": persona_card(r),
    })
    el = time.perf_counter() - t0

    # 3) save (풀 페르소나 27 컬럼 모두 포함)
    rec = {
        "persona_uuid": r["uuid"],
        "sex": r["sex"], "age": int(r["age"]),
        "marital_status": r["marital_status"],
        "family_type": r["family_type"], "housing_type": r["housing_type"],
        "education_level": r["education_level"],
        "bachelors_field": r["bachelors_field"],
        "occupation": r["occupation"],
        "province": r["province"], "district": r["district"],
        "persona_summary": r["persona"],
        "vote": p["vote"], "reason": p["reason"],
        "model": f"{provider}/{model_name}",
        "elapsed_sec": round(el, 3),
        "response_file": "",
        "professional_persona": r["professional_persona"],
        "sports_persona": r["sports_persona"],
        "arts_persona": r["arts_persona"],
        "travel_persona": r["travel_persona"],
        "culinary_persona": r["culinary_persona"],
        "family_persona": r["family_persona"],
        "cultural_background": r["cultural_background"],
        "skills_and_expertise": r["skills_and_expertise"],
        "hobbies_and_interests": r["hobbies_and_interests"],
        "career_goals_and_ambitions": r["career_goals_and_ambitions"],
    }
    df_rec = pd.DataFrame([rec])
    if RESULTS_PATH.exists():
        df_rec.to_csv(RESULTS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_rec.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")

    # 4) pretty print
    if verbose:
        labels = [
            ("uuid",       rec["persona_uuid"]),
            ("성별",        rec["sex"]),
            ("연령",        f'{rec["age"]}세'),
            ("혼인상태",     rec["marital_status"]),
            ("가구형태",     rec["family_type"]),
            ("주거",        rec["housing_type"]),
            ("학력",        rec["education_level"]),
            ("전공",        rec["bachelors_field"]),
            ("직업",        rec["occupation"]),
            ("지역",        f'{rec["province"]} {rec["district"]}'),
            ("요약",        rec["persona_summary"]),
            ("─" * 8,       "─" * 8),
            ("투표",        rec["vote"]),
            ("이유",        rec["reason"]),
            ("모델",        rec["model"]),
            ("소요시간",     f'{rec["elapsed_sec"]}s'),
        ]
        width = max(len(k) for k, _ in labels)
        for k, v in labels:
            print(f"{k:<{width}}  {v}")
    return rec

def run_loop(provider: str, model_name: str | None, n: int):
    """n회 반복. 매 호출 결과는 1줄 요약으로 출력. 마지막 1건만 항목별 풀 출력."""
    print(f"=== {provider}/{model_name or MODEL} × {n} ===")
    for i in range(n):
        rec = run_one(provider, model_name, verbose=(i == n - 1))
        if i < n - 1:
            print(f"[{i+1:>3}/{n}] {rec['sex']} {rec['age']}세 {rec['province']} "
                  f"{rec['occupation']:<12} → {rec['vote']:<10} ({rec['elapsed_sec']}s)")

In [ ]:
# [Cell 9] OpenAI — 반복 회수 N 조정 후 실행
N = 1
run_loop("openai", MODEL, N)

In [ ]:
# [Cell 10] EXAONE 4.0 32B (LG, 한국어 특화) — 반복 회수 N 조정 후 실행
N = 20
run_loop("ollama", "ingu627/exaone4.0:32b", N)

In [ ]:
# [Cell 11] Qwen3 32B (Apache 2.0, 다국어) — 반복 회수 N 조정 후 실행
N = 20
run_loop("ollama", "qwen3:32b", N)

In [ ]:
# [Cell 12] Gemma 3 27B (Google, 140개 언어) — 반복 회수 N 조정 후 실행
N = 20
run_loop("ollama", "gemma3:27b", N)

## 다음 단계

`sample-persona` ~ `save` 셀을 반복 실행하면 한 명씩 `vote_results_all.csv`에 누적됩니다.

- OpenAI 모델은 [Cell 1]의 `MODEL` 변수, 로컬 모델은 [Cell 10~12]의 model_name 인자에서 변경
- `temperature=0.7` 로 다양성 확보. 결정적 결과 원하면 `0`으로
- N=500/1000 배치는 별도 노트북에서 루프·재시도·층화추출 추가 예정